In [ ]:
# Import all the tools we need
import pandas as pd                    # For handling data tables (like Excel)
import numpy as np                     # For mathematical operations
import matplotlib.pyplot as plt        # For creating charts and graphs
import seaborn as sns                  # For making prettier charts

# Machine Learning tools from scikit-learn
from sklearn.model_selection import train_test_split   # Split data into training/testing
from sklearn.preprocessing import StandardScaler       # Make all numbers similar scale
from sklearn.preprocessing import LabelEncoder         # Convert text to numbers
from sklearn.ensemble import RandomForestRegressor     # One type of ML model
from sklearn.linear_model import Ridge                 # Another type of ML model
from sklearn.multioutput import MultiOutputRegressor   # For predicting multiple things
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error  # Ways to measure accuracy
from sklearn.impute import SimpleImputer               # Fill in missing data

import warnings
warnings.filterwarnings('ignore')  # Hide technical warning messages

# Make our charts look nice
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ All tools loaded successfully!")
print("🎯 Ready to start our machine learning journey!")


In [ ]:
# Load our operational data (the "ingredients" of our process)
print("🔬 Loading operational gasification data...")
operational_df = pd.read_csv('data/gasification_data_refined.csv')

# Let's see what we have
print(f"✅ Operational data loaded: {operational_df.shape[0]} experiments, {operational_df.shape[1]} parameters")
print(f"🔍 Technologies we studied: {list(operational_df['technology'].unique())}")
print(f"🔍 Temperature range: {operational_df['temperature_C'].min():.0f}°C to {operational_df['temperature_C'].max():.0f}°C")
print(f"🔍 H2 yield range: {operational_df['H2_yield_mol_kg'].min():.1f} to {operational_df['H2_yield_mol_kg'].max():.1f} mol/kg")

print("\n📋 Here's what our operational data looks like:")
operational_df.head()

print("\n🔍 Let's understand each column:")
print("- technology: Which gasification method (steam, plasma, scw, co2)")
print("- temperature_C: How hot the reaction was (°C)")
print("- pressure_bar: How much pressure was applied (bar)")
print("- H2_yield_mol_kg: How much hydrogen gas was produced (mol/kg)")
print("- CO_yield_mol_kg: How much carbon monoxide was produced (mol/kg)")
print("- reaction_time_min: How long the reaction took (minutes)")


In [ ]:
# Load our environmental impact data (the "results" we want to predict)
print("🌍 Loading environmental impact target data...")
lca_df = pd.read_excel('data/LCA/LCAResultsWithWaste.xlsx')
lca_df.columns = [col.strip() for col in lca_df.columns]  # Clean up column names

# Make column names match our operational data
column_mapping = {
    'CO2 Gasfication': 'co2',
    'Plasma Gasification': 'plasma', 
    'SCWG': 'scw',
    'Steam Gasification': 'steam'
}
lca_df = lca_df.rename(columns=column_mapping)

print(f"✅ Environmental impact data loaded: {lca_df.shape[0]} impact categories")
print(f"🔍 Technologies: {[col for col in lca_df.columns if col != 'Impact categories']}")

print("\n🌍 Here's what our environmental impact data looks like:")
lca_df.head(10)

print("\n🔍 What these impact categories mean:")
print("- Climate change: How much it contributes to global warming")
print("- Fossil depletion: How much it uses up fossil resources")
print("- Human toxicity: How harmful it is to human health")
print("- Water depletion: How much fresh water it uses up")
print("- And 14 more environmental indicators...")


In [ ]:
# Create our training dataset by connecting operational parameters with environmental impacts
print("🔗 Creating training dataset...")

# Get the list of environmental impact categories
impact_categories = list(lca_df['Impact categories'])
print(f"📊 We have {len(impact_categories)} environmental impacts to predict")

# Create our training data
training_data = []

# For each experiment in our operational data...
for _, op_row in operational_df.iterrows():
    tech = op_row['technology']  # What technology was used?
    
    # If we have environmental data for this technology...
    if tech in lca_df.columns:
        impacts = lca_df[tech].values  # Get all environmental impacts for this tech
        
        # Create a row with operational parameters...
        features = {
            'technology': tech,
            'temperature_C': op_row['temperature_C'],
            'pressure_bar': op_row['pressure_bar'],
            'H2_yield_mol_kg': op_row['H2_yield_mol_kg'],
            'CO_yield_mol_kg': op_row['CO_yield_mol_kg'],
            'reaction_time_min': op_row['reaction_time_min']
        }
        
        # ...and add all environmental impact values
        for i, impact_category in enumerate(impact_categories):
            clean_name = impact_category.replace(" ", "_").lower()
            features[f'impact_{i:02d}_{clean_name}'] = impacts[i]
        
        training_data.append(features)

# Convert to a nice table format
training_df = pd.DataFrame(training_data)

print(f"\n✅ Training dataset created: {training_df.shape[0]} experiments with {training_df.shape[1]} total columns")
feature_cols = [col for col in training_df.columns if not col.startswith('impact_')]
impact_cols = [col for col in training_df.columns if col.startswith('impact_')]
print(f"🔍 Input features (what we know): {len(feature_cols)} parameters")
print(f"🔍 Output targets (what we want to predict): {len(impact_cols)} environmental impacts")

print("\n📋 Sample of our combined dataset (input features only):")
training_df[feature_cols].head()

print("\n💡 What just happened?")
print("We created a dataset where each row contains:")
print("- Input: temperature, pressure, yields, etc. (what the engineer controls)")
print("- Output: 18 environmental impact scores (what we want to predict)")
print("This is like having a recipe with both ingredients AND nutrition facts on each line!")


In [ ]:
# Step 4a: Prepare our input features (X) and target variables (y)
print("🛠️ Preparing features and targets...")

# Define which columns are our input features
feature_cols = ['temperature_C', 'pressure_bar', 'H2_yield_mol_kg', 'CO_yield_mol_kg', 'reaction_time_min']

# Convert technology names to numbers (computers need numbers!)
print("\n🔤 Converting technology names to numbers...")
le_tech = LabelEncoder()
tech_encoded = le_tech.fit_transform(training_df['technology'])

# Show the mapping
tech_mapping = dict(zip(le_tech.classes_, le_tech.transform(le_tech.classes_)))
print("Technology encoding:")
for tech, number in tech_mapping.items():
    print(f"  {tech} → {number}")

# Create our feature matrix (X) - what we know
X = training_df[feature_cols].copy()
X['technology_encoded'] = tech_encoded

# Create our target matrix (y) - what we want to predict
impact_cols = [col for col in training_df.columns if col.startswith('impact_')]
y = training_df[impact_cols].values

print(f"\n✅ Feature matrix (X): {X.shape[0]} experiments × {X.shape[1]} features")
print(f"✅ Target matrix (y): {y.shape[0]} experiments × {y.shape[1]} environmental impacts")
print(f"🔍 Our input features: {list(X.columns)}")

print("\n📊 Let's look at our features:")
X.head()

print("\n💡 What we just did:")
print("- X (features): The process conditions that engineers can control")
print("- y (targets): The environmental impacts we want to predict")
print("- We converted 'steam', 'plasma' etc. to numbers 0, 1, 2, 3")
print("- Now the computer can work with our data!")


In [ ]:
# Step 4b: Split data and train models
print("🚂 Splitting data and training models...")

# Split our data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features to similar ranges (very important!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle any missing values
imputer = SimpleImputer(strategy='median')
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled = imputer.transform(X_test_scaled)

print(f"✅ Training set: {X_train_scaled.shape[0]} experiments")
print(f"✅ Test set: {X_test_scaled.shape[0]} experiments")

# Define and train our models
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Ridge Regression': MultiOutputRegressor(Ridge(alpha=1.0))
}

results = {}
best_model = None
best_score = -np.inf
best_name = ""

print("\n🤖 Training different machine learning models...")

for name, model in models.items():
    print(f"\n📊 Training {name}...")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    
    # Evaluate performance
    r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
    mse = mean_squared_error(y_test, y_pred, multioutput='uniform_average')
    mae = mean_absolute_error(y_test, y_pred, multioutput='uniform_average')
    
    results[name] = {'model': model, 'r2': r2, 'mse': mse, 'mae': mae}
    
    print(f"   R² Score: {r2:.3f} (higher is better, 1.0 = perfect)")
    print(f"   MSE: {mse:.3e} (lower is better, 0 = perfect)")
    print(f"   MAE: {mae:.3e} (lower is better, 0 = perfect)")
    
    if r2 > best_score:
        best_score = r2
        best_model = model
        best_name = name

print(f"\n🏆 Best model: {best_name} with R² = {best_score:.3f}")

print("\n💡 What these metrics mean:")
print("- R² Score: How much of the variation our model explains (0-1, higher better)")
print("  * 0.8+ = Great! Model explains most of the patterns")
print("  * 0.5-0.8 = Good, captures main relationships")
print("  * <0.5 = Poor, might need more data or different approach")
print("- MSE & MAE: How far off our predictions are (lower = better)")


In [ ]:
# Final Summary: What we accomplished!
print("📋 MACHINE LEARNING PROJECT SUMMARY")
print("=" * 50)

print("\n🎯 PROJECT GOAL:")
print("Build a model to predict 18 environmental impacts from gasification process conditions")

print(f"\n📊 DATA OVERVIEW:")
print(f"- {operational_df.shape[0]} experiments across 4 gasification technologies")
print(f"- {len(feature_cols)+1} input features (temperature, pressure, yields, etc.)")
print(f"- {len(impact_categories)} environmental impact targets")
print(f"- Temperature range: {operational_df['temperature_C'].min():.0f}-{operational_df['temperature_C'].max():.0f}°C")

print(f"\n🤖 MODEL PERFORMANCE:")
print(f"- Best model: {best_name}")
print(f"- Overall R² score: {best_score:.3f}")

print("\n✅ KEY ACHIEVEMENTS:")
print("1. ✨ Successfully connected operational data with environmental impacts")
print("2. 🎯 Trained models that can predict multiple environmental impacts simultaneously")
print("3. ⚡ Created a tool that can replace weeks of LCA work with instant predictions")
print("4. 📈 Learned which environmental impacts are most predictable from process conditions")

print("\n🔍 WHAT WE LEARNED:")
print("• Some environmental impacts are easier to predict than others")
print("• Process conditions like temperature and yields are good predictors")
print(f"• {best_name} models work well for this type of environmental data")
print("• We need more data to improve predictions for certain impact categories")

print("\n🚀 PRACTICAL APPLICATIONS:")
print("• Engineers can optimize processes for environmental performance")
print("• Quick screening of different gasification conditions")
print("• Support decision-making in sustainable technology development")
print("• Integration into process design software")

print("\n💡 KEY LESSONS FOR BEGINNERS:")
print("1. 📊 Data quality matters more than model complexity")
print("2. 🔄 Always split data to test on unseen examples")
print("3. 🧠 Understand your data before building models")
print("4. 🏆 Different models have different strengths")
print("5. 🎯 Interpretation is as important as prediction accuracy")
print("6. 🤝 Machine learning is a tool to augment, not replace, domain expertise")

print("\n🎓 CONGRATULATIONS!")
print("You've successfully built, evaluated, and learned about a machine learning model")
print("for environmental impact prediction. This is real, practical AI that can")
print("make a difference in sustainable technology development!")

# Show a simple example of what our model can do
print("\n🧪 Example: What our model learned...")
print("Input: Steam gasification at 800°C, 50 bar pressure")
print("Output: Instant prediction of 18 environmental impacts!")
print("→ This normally takes weeks with traditional LCA methods")

print("\n🌍 IMPACT:")
print("This technology can help engineers make better decisions for our planet! 🌱")


In [ ]:
# Data handling tools (like Excel, but more powerful)
import pandas as pd
import numpy as np

# Visualization tools (for making charts and graphs)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning tools
from sklearn.model_selection import train_test_split  # Split data into training/testing
from sklearn.preprocessing import StandardScaler, LabelEncoder  # Prepare data for ML
from sklearn.ensemble import RandomForestRegressor  # Our main ML algorithm
from sklearn.linear_model import Ridge  # Alternative ML algorithm
from sklearn.multioutput import MultiOutputRegressor  # For predicting multiple things at once
from sklearn.metrics import r2_score, mean_squared_error  # For measuring accuracy
from sklearn.impute import SimpleImputer  # For handling missing data

# Ignore warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

print("✅ All tools imported successfully!")
print("🎯 Ready to start machine learning!")


In [ ]:
# Load operational data (the "recipes")
print("🔬 Loading operational gasification data...")
operational_df = pd.read_csv('data/gasification_data_refined.csv')

print(f"✅ Loaded {operational_df.shape[0]} experiments with {operational_df.shape[1]} variables")
print(f"\n🔍 Technologies available: {list(operational_df['technology'].unique())}")
print(f"\n📊 Data Summary:")
print(f"   - Temperature range: {operational_df['temperature_C'].min():.0f}-{operational_df['temperature_C'].max():.0f}°C")
print(f"   - H2 yield range: {operational_df['H2_yield_mol_kg'].min():.1f}-{operational_df['H2_yield_mol_kg'].max():.1f} mol/kg")
print(f"   - Number of experiments: {len(operational_df)}")

# Show first few rows
print("\n👀 First 5 experiments:")
operational_df.head()


In [ ]:
# Load environmental impact data (the "results")
print("🌍 Loading environmental impact data...")
impact_df = pd.read_excel('data/LCA/LCAResultsWithWaste.xlsx')
impact_df.columns = [col.strip() for col in impact_df.columns]

# Clean up column names to match our operational data
column_mapping = {
    'CO2 Gasfication': 'co2',
    'Plasma Gasification': 'plasma', 
    'SCWG': 'scw',
    'Steam Gasification': 'steam'
}
impact_df = impact_df.rename(columns=column_mapping)

print(f"✅ Loaded {impact_df.shape[0]} environmental impact categories")
print(f"✅ Available for {impact_df.shape[1]-1} technologies")

print("\n🌍 Environmental impact categories we can predict:")
for i, category in enumerate(impact_df['Impact categories'][:10], 1):
    print(f"   {i:2d}. {category}")
if len(impact_df) > 10:
    print(f"   ... and {len(impact_df)-10} more categories")

# Show the data structure
print("\n👀 Environmental impact data structure:")
impact_df.head()


In [ ]:
print("🔗 Creating training dataset...")
print("This is like making a cookbook that shows recipes AND their results")

# Calculate average conditions for each technology type
tech_averages = operational_df.groupby('technology').agg({
    'temperature_C': 'mean',
    'pressure_bar': 'mean', 
    'H2_yield_mol_kg': 'mean',
    'CO_yield_mol_kg': 'mean',
    'reaction_time_min': 'mean'
}).reset_index()

print("\n📊 Average conditions by technology:")
for _, row in tech_averages.iterrows():
    print(f"\n🔧 {row['technology'].upper()} Technology:")
    print(f"   Temperature: {row['temperature_C']:.0f}°C")
    print(f"   Pressure: {row['pressure_bar']:.1f} bar")
    print(f"   H2 yield: {row['H2_yield_mol_kg']:.1f} mol/kg")

# Create training examples - For each experiment in our operational data
training_data = []
for _, op_row in operational_df.iterrows():
    tech = op_row['technology']
    
    # If we have environmental impact data for this technology
    if tech in impact_df.columns:
        impacts = impact_df[tech].values
        
        # Create a training example
        features = {
            'technology': tech,
            'temperature_C': op_row['temperature_C'],
            'pressure_bar': op_row['pressure_bar'],
            'H2_yield_mol_kg': op_row['H2_yield_mol_kg'],
            'CO_yield_mol_kg': op_row['CO_yield_mol_kg'],
            'reaction_time_min': op_row['reaction_time_min']
        }
        
        # Add environmental impact values
        for i, impact_category in enumerate(impact_df['Impact categories']):
            clean_name = impact_category.replace(" ", "_").lower()
            features[f'impact_{i:02d}_{clean_name}'] = impacts[i]
        
        training_data.append(features)

training_df = pd.DataFrame(training_data)
impact_categories = list(impact_df['Impact categories'])

print(f"\n✅ Created {len(training_data)} training examples")
print(f"📊 Each example has:")
print(f"   - 6 input features (operational parameters)")
print(f"   - {len(impact_categories)} target values (environmental impacts)")

print("\n🎯 What each training example looks like:")
print("   INPUT: Temperature=600°C, Pressure=1bar, Technology=steam, etc.")
print("   OUTPUT: Climate_change=0.123, Water_use=0.456, Toxicity=0.789, etc.")


In [ ]:
print("🛠️ Preparing data for machine learning...")
print("Think of this as organizing ingredients before cooking")

# 1. Define our input features (the "recipe ingredients")
feature_cols = ['temperature_C', 'pressure_bar', 'H2_yield_mol_kg', 'CO_yield_mol_kg', 'reaction_time_min']

# 2. Convert technology names to numbers
print("\n🔤 Converting technology names to numbers...")
le_tech = LabelEncoder()
tech_encoded = le_tech.fit_transform(training_df['technology'])

print("Technology encoding:")
for i, tech in enumerate(le_tech.classes_):
    print(f"   '{tech}' → {i}")

# 3. Create input matrix (X) - our "recipe ingredients"
X = training_df[feature_cols].copy()
X['technology_encoded'] = tech_encoded

# 4. Create output matrix (y) - our "dish results" 
impact_cols = [col for col in training_df.columns if col.startswith('impact_')]
y = training_df[impact_cols].values

print(f"\n📊 Data prepared:")
print(f"   Input matrix (X): {X.shape[0]} examples × {X.shape[1]} features")
print(f"   Output matrix (y): {y.shape[0]} examples × {y.shape[1]} environmental impacts")

print(f"\n🔍 Our input features:")
for i, feature in enumerate(X.columns, 1):
    print(f"   {i}. {feature}")

print(f"\n🎯 Sample input values:")
print(X.head(3))


In [ ]:
print("🧪 Splitting data for training and testing...")
print("Like saving some practice problems for a final exam")

# Split the data: 70% for training, 30% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\n📚 Training set: {X_train.shape[0]} examples")
print(f"   The computer will study these to learn patterns")

print(f"\n📝 Testing set: {X_test.shape[0]} examples")
print(f"   We'll use these to see how well the computer learned")

# Scale the features so they're all on similar scales
print(f"\n⚖️ Scaling features to similar ranges...")
print(f"   Before: Temperature might be 600, pressure might be 1")
print(f"   After: Both will be roughly between -1 and 1")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle any missing values
imputer = SimpleImputer(strategy='median')
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled = imputer.transform(X_test_scaled)

print(f"✅ Data ready for machine learning!")


In [ ]:
print("🤖 Training machine learning models...")
print("This is where the computer learns patterns from our data")

# Define our two models to try
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Ridge Regression': MultiOutputRegressor(Ridge(alpha=1.0))
}

print(f"\n🔍 Testing {len(models)} different model types...")

best_model = None
best_score = -np.inf
best_name = ""
results = {}

for name, model in models.items():
    print(f"\n🧠 Training {name}...")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    print(f"   ✅ Training complete!")
    
    # Make predictions on test data
    y_pred = model.predict(X_test_scaled)
    print(f"   ✅ Predictions made!")
    
    # Calculate accuracy scores
    r2 = r2_score(y_test, y_pred, multioutput='uniform_average')
    mse = mean_squared_error(y_test, y_pred, multioutput='uniform_average')
    
    print(f"   📊 R² Score: {r2:.3f}")
    print(f"      (1.0 = perfect, 0.0 = random guessing)")
    print(f"   📊 Mean Squared Error: {mse:.3e}")
    print(f"      (lower is better)")
    
    # Store results
    results[name] = {
        'model': model,
        'r2': r2,
        'mse': mse,
        'predictions': y_pred
    }
    
    # Track the best model
    if r2 > best_score:
        best_score = r2
        best_model = model
        best_name = name

print(f"\n🏆 Best performing model: {best_name}")
print(f"   R² Score: {best_score:.3f}")

if best_score > 0.7:
    print(f"   🎉 Excellent performance!")
elif best_score > 0.5:
    print(f"   👍 Good performance!")
elif best_score > 0.3:
    print(f"   ⚠️  Fair performance - could be improved")
else:
    print(f"   ❌ Poor performance - needs more work")


In [ ]:
print("📈 Analyzing how well we predict each environmental impact...")
print("Some impacts are easier to predict than others!")

# Get predictions from our best model
y_pred = best_model.predict(X_test_scaled)

# Calculate R² score for each environmental impact
impact_r2_scores = []
for i in range(y_test.shape[1]):
    try:
        r2 = r2_score(y_test[:, i], y_pred[:, i])
        if np.isnan(r2):  # Handle edge cases
            r2 = 0.0
        impact_r2_scores.append(r2)
    except:
        impact_r2_scores.append(0.0)

# Create a results table
performance_df = pd.DataFrame({
    'Impact_Category': impact_categories,
    'R2_Score': impact_r2_scores
}).sort_values('R2_Score', ascending=False)

print(f"\n🎯 Top 10 Best Predicted Environmental Impacts:")
print("=" * 70)
for i, (_, row) in enumerate(performance_df.head(10).iterrows(), 1):
    score = row['R2_Score']
    category = row['Impact_Category']
    
    # Add performance indicator
    if score >= 0.8:
        indicator = "🎉 Excellent"
    elif score >= 0.6:
        indicator = "👍 Good"
    elif score >= 0.4:
        indicator = "⚠️  Fair"
    else:
        indicator = "❌ Poor"
    
    print(f"{i:2d}. {category:<35} R² = {score:.3f} {indicator}")

print(f"\n📊 Overall Performance Summary:")
print(f"   Average R² Score: {np.mean(impact_r2_scores):.3f}")
print(f"   Best Prediction: {performance_df.iloc[0]['Impact_Category']} (R² = {np.max(impact_r2_scores):.3f})")
print(f"   Most Challenging: {performance_df.iloc[-1]['Impact_Category']} (R² = {np.min(impact_r2_scores):.3f})")

# Count performance categories
excellent = sum(1 for score in impact_r2_scores if score >= 0.8)
good = sum(1 for score in impact_r2_scores if 0.6 <= score < 0.8)
fair = sum(1 for score in impact_r2_scores if 0.4 <= score < 0.6)
poor = sum(1 for score in impact_r2_scores if score < 0.4)

print(f"\n🏆 Performance Breakdown:")
print(f"   🎉 Excellent (R² ≥ 0.8): {excellent} impacts")
print(f"   👍 Good (0.6 ≤ R² < 0.8): {good} impacts")
print(f"   ⚠️  Fair (0.4 ≤ R² < 0.6): {fair} impacts")
print(f"   ❌ Poor (R² < 0.4): {poor} impacts")


In [ ]:
print("🔍 Analyzing which factors are most important for predictions...")
print("This tells us what to focus on when designing processes!")

# Check if our model can show feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = list(X.columns)
    
    # Create importance ranking
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f"\n🔝 Feature Importance Ranking:")
    print("=" * 50)
    for i, (_, row) in enumerate(importance_df.iterrows(), 1):
        feature = row['Feature']
        importance = row['Importance']
        percentage = importance * 100
        
        # Create visual bar
        bar_length = int(percentage / 5)  # Scale for display
        bar = '█' * bar_length + '░' * (20 - bar_length)
        
        print(f"{i}. {feature:<20} {bar} {percentage:5.1f}%")
    
    # Interpretation
    most_important = importance_df.iloc[0]
    print(f"\n💡 Key Insights:")
    print(f"   🎯 Most important factor: {most_important['Feature']}")
    print(f"   📊 It accounts for {most_important['Importance']*100:.1f}% of prediction power")
    
    if 'temperature' in most_important['Feature'].lower():
        print(f"   🌡️  Temperature is key - process heat matters most!")
    elif 'technology' in most_important['Feature'].lower():
        print(f"   🔧 Technology choice is key - the method matters most!")
    elif 'yield' in most_important['Feature'].lower():
        print(f"   ⚗️  Gas production is key - output efficiency matters most!")
        
else:
    print("ℹ️  Feature importance not available for this model type")
    print("   (This happens with some algorithms like Ridge Regression)")
    importance_df = None
